# 00 - Epoch extraction

Turns the raw Faller et al. (2019) `.mat` recordings into the two ring-locked
epoch sets every later notebook depends on.

| output | contents | used by |
|---|---|---|
| `data/ring_events.pkl` | fixed 2 s (512-sample) pre-crossing windows | decoder training and CV |
| `data/ring_events_online.pkl` | variable-length windows tiling each trial | continuous arousal decoding |
| `data/trial_events.pkl` | per-trial ring-crossing times | trial annotation |

Raw recordings are not redistributed here. Download them from
[IEEE DataPort](https://doi.org/10.21227/rn3e-bp31) and point `RAW_DIR` at the
folder of `.mat` files.

**This notebook only needs to be run once**, and takes roughly 20 minutes. The
epoch pickles are large (~4 GB each). Skip to notebook 01 if you already have them.

In [1]:
import sys
sys.path.insert(0, "..")   # or: pip install -e .. to import `arousal` directly

import numpy as np

from arousal import config as cfg
from arousal import extract

RAW_DIR = "../../dataset"       # folder containing S01_*.mat ... S20_*.mat
OVERWRITE = False               # set True to regenerate existing pickles

print(f"{len(cfg.SUBJECTS)} subjects: {cfg.SUBJECTS}")
print(f"conditions: {cfg.CONDITIONS}")

16 subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 17, 18, 19, 20]
conditions: {0: 'silence-openloop', 1: 'silence', 2: 'half', 3: 'bci'}


## Fixed-length epochs

Each epoch is the 2 s of physiology immediately preceding a ring crossing.
Channels are z-scored against that subject's pre-experiment resting recording
before epoching; note that only the EEG and peripheral channels are normalised,
so the feedback and behavioural channels remain in raw units.

In [2]:
if OVERWRITE or not cfg.RING_EVENTS.exists():
    epochs, open_lengths = extract.build_ring_events(RAW_DIR, online=False)
    extract.write_pickle({"events": epochs, "lengths": open_lengths}, cfg.RING_EVENTS)
    print(f"wrote {len(epochs)} epochs -> {cfg.RING_EVENTS.name}")
else:
    print(f"{cfg.RING_EVENTS.name} already exists; set OVERWRITE=True to rebuild")

ring_events.pkl already exists; set OVERWRITE=True to rebuild


## Variable-length epochs

Here each epoch spans the interval between consecutive crossings, so that
concatenating a trial's epochs reconstructs it as one continuous recording.
This is what the sliding-window decoder runs over.

In [3]:
if OVERWRITE or not cfg.RING_EVENTS_ONLINE.exists():
    epochs, _ = extract.build_ring_events(RAW_DIR, online=True)
    extract.write_pickle(epochs, cfg.RING_EVENTS_ONLINE)
    print(f"wrote {len(epochs)} epochs -> {cfg.RING_EVENTS_ONLINE.name}")
else:
    print(f"{cfg.RING_EVENTS_ONLINE.name} already exists")

ring_events_online.pkl already exists


## Trial-level ring crossings

In [4]:
if OVERWRITE or not cfg.TRIAL_EVENTS.exists():
    rows = extract.build_trial_events(RAW_DIR)
    extract.write_pickle(rows, cfg.TRIAL_EVENTS)
    print(f"wrote {len(rows)} trials -> {cfg.TRIAL_EVENTS.name}")
else:
    print(f"{cfg.TRIAL_EVENTS.name} already exists")

trial_events.pkl already exists


## Sanity check

In [5]:
from arousal import data as D

ring = D.load_ring_epochs()
online = D.load_ring_epochs_online()

print(f"fixed-length : {len(ring):>6} epochs, shape {ring['data'].iloc[0].shape}")
print(f"variable     : {len(online):>6} epochs")
print("\nepochs per condition (fixed-length):")
for c, n in ring["condition"].value_counts().sort_index().items():
    print(f"  {c} {cfg.CONDITIONS[c]:<20} {n:>5}")
print("\nring sizes:", ring["ring_size"].value_counts().to_dict())
print("task-demand labels:", ring["label"].value_counts().to_dict())

fixed-length :  13481 epochs, shape (144, 512)
variable     :  13969 epochs

epochs per condition (fixed-length):
  0 silence-openloop      4088
  1 silence               3078
  2 half                  3128
  3 bci                   3187

ring sizes: {'large': 7230, 'medium': 5099, 'small': 1152}
task-demand labels: {0: 7230, 1: 6251}
